# Faruq-v3 PCL1 — PCLDet learned-prototype screening

Broad-search seed 42 and predecessor control for APCL. PCL1 transfers PCLDet cosine prototype similarity + ProtoCL learned prototypes (paper Eqs. 1, 3, 4) to positive YOLO26 one-to-many dense assignments. CBS/ReDet/RPN are intentionally excluded. Validation only; locked test is never restored or opened.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, shutil, subprocess, sys, tarfile, time
from pathlib import Path

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/pcldet-prototype-baseline-screening'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
clone = ['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)]
for attempt in range(1, 4):
    result = subprocess.run(clone)
    if result.returncode == 0: break
    if REPO.exists(): shutil.rmtree(REPO)
    if attempt == 3: raise RuntimeError('Git clone gagal tiga kali.')
    time.sleep(2)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
for module_name in list(sys.modules):
    if module_name == 'coffee_detector' or module_name.startswith('coffee_detector.'):
        sys.modules.pop(module_name, None)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
print('REPO:', REPO)
print('BRANCH:', BRANCH)

In [ ]:
import torch
from coffee_detector.drive_project import require_project_artifact, resolve_drive_project_root

assert torch.cuda.is_available(), 'Aktifkan T4 GPU.'
PROJECT_ROOT = resolve_drive_project_root(required_relative_paths=(
    'bundles/faruq-development-v3-grouped.tar',
    'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json',
    'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json',
))
ARCHIVE = require_project_artifact(PROJECT_ROOT, 'bundles/faruq-development-v3-grouped.tar')
D0_CHECKPOINT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-yolo26n-baseline-v1/D0_seed42/weights/best.pt')
D0FT_REPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/D0FT_seed42_val.json')
ACMC1_REPORT = require_project_artifact(PROJECT_ROOT, 'experiments/faruq-v3-acmc-optimization-control-v1/val_reports/acmc1_optimization_control_seed42.json')
DATA_ROOT = Path('/content/faruq-development-v3-grouped')
GROUPED_SUMMARY = DATA_ROOT / 'faruq_grouped_summary.json'
OUTPUT_ROOT = PROJECT_ROOT / 'experiments/faruq-v3-pcldet-prototype-search-v1'
if not GROUPED_SUMMARY.is_file():
    with tarfile.open(ARCHIVE, 'r') as archive: archive.extractall('/content', filter='data')
assert GROUPED_SUMMARY.is_file(), GROUPED_SUMMARY
assert not (DATA_ROOT / 'test').exists(), 'Test tidak boleh tersedia.'
print('GPU       :', torch.cuda.get_device_name(0))
print('PROJECT   :', PROJECT_ROOT)
print('D0        :', D0_CHECKPOINT)
print('D0FT      :', D0FT_REPORT)
print('ACMC1     :', ACMC1_REPORT)
print('OUTPUT    :', OUTPUT_ROOT)
last = OUTPUT_ROOT / 'PCL1_seed42/weights/last.pt'
best = OUTPUT_ROOT / 'PCL1_seed42/weights/best.pt'
print('PCL1      :', 'COMPLETE' if best.is_file() else ('RESUME' if last.is_file() else 'START'))

In [ ]:
command = [sys.executable, '-m', 'pytest', '-q', 'tests/test_pcl.py']
print('STATIC CHECK:', ' '.join(command))
subprocess.run(command, cwd=REPO, check=True)
print('PASS: native inference unchanged; learned-prototype Eq.3/gradient contract verified.')

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.experiments.run_faruq_v3_pcl_screening',
    '--data-root', str(DATA_ROOT),
    '--grouped-summary', str(GROUPED_SUMMARY),
    '--d0-checkpoint', str(D0_CHECKPOINT),
    '--d0ft-report', str(D0FT_REPORT),
    '--acmc1-report', str(ACMC1_REPORT),
    '--output-root', str(OUTPUT_ROOT),
    '--seed', '42', '--device', '0', '--authorize-training',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.run(command, cwd=REPO, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(process.stdout, end='', flush=True)
if process.returncode != 0:
    raise RuntimeError(f'PCL1 gagal dengan return code {process.returncode}; traceback lengkap tercetak di atas.')

In [ ]:
import json, pandas as pd
from IPython.display import display

SUMMARY = OUTPUT_ROOT / 'val_reports/pcl1_seed42_screening.json'
assert SUMMARY.is_file(), f'PCL1 belum selesai: {SUMMARY}'
result = json.loads(SUMMARY.read_text(encoding='utf-8'))
assert result['evaluation_split'] == 'val'
assert result['test_images_accessed'] is False
rows = [{'model': name, **metrics} for name, metrics in result['results'].items()]
metrics = ('macro_map50_95', 'bottom3_class_map50_95', 'worst_class_map50_95')
display(pd.DataFrame(rows).style.format({name: '{:.2%}' for name in metrics}))
print('PCL1 vs D0FT :', result['deltas']['PCL1_vs_D0FT'])
print('PCL1 vs ACMC1:', result['deltas']['PCL1_vs_ACMC1'])
print('BOUNDARY      :', result['adaptation_boundary'])
print('CRITERIA      :', result['criteria'])
print('DECISION      :', result['decision'])
print('NEXT          :', result['next_action'])
print('SUMMARY       :', SUMMARY)
print('PCL1 adalah predecessor/method control untuk APCL; test tetap terkunci.')